# 03 - EELS analysis of the lamella dataset

Same workflow as notebook 01, on a different dataset: a lamella whose spectra
cover 138 to 650 eV. The measurable edges there belong to Si, S, C, Ca and O -
the Ca-L2,3 edge at 346 eV is the strongest, and this dataset comes with six
Ca reference spectra for fine-structure analysis.

**This dataset is not part of the participant ZIP** (which holds `nanopore`
only). Notebook 03 is meant for instructors.

**Work through notebook 01 first** - the individual steps are explained there,
only the differences are described here.

In [ ]:
# Interactive plots (zoom, click a pixel to see its spectrum).
# If plots stay blank or nothing appears at all:
# replace this line with  %matplotlib inline  and restart the kernel.
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import hyperspy.api as hs
import exspy  # must be imported, otherwise HyperSpy does not know the EELS/EDX signal types

# Finds the data regardless of operating system (see workshop_data.py)
from workshop_data import load, load_standards

print("HyperSpy", hs.__version__, "| exspy", exspy.__version__)

## 1. Load the data

In [ ]:
signal = load("lamella_eels_highloss", signal_type="EELS")
ll = load("lamella_eels_lowloss", signal_type="EELS")

signal.plot()

In [ ]:
load("lamella_adf").plot()

## 2. Align the zero-loss peak

In [ ]:
ll.align_zero_loss_peak(also_align=[signal], signal_range=(-10.0, 10.0))

## 3. Build the model

Note the binning of **8x8** here. That is a deliberate trade-off: spatial
resolution against runtime. If you have time, try 4x4.

In [ ]:
# Ga, W and Pt were in this list originally. Their nearest edges sit at 1115,
# 1809 and 2122 eV - but this acquisition only reaches 650 eV. exspy dropped
# them silently. What remains are elements whose edges really do fall inside
# the measured range of 138-650 eV.
signal.add_elements(["Si", "S", "C", "Ca", "O"])

signal_binned = signal.rebin(scale=[8, 8, 1])

# The background window has to sit BEFORE the first edge. The first edge is
# Si-L1 at 150 eV and the acquisition starts at 138 eV, so there is not much room.
# (The original had (70, 96) here, copied from notebook 01. That range lies
# completely outside this data - without raising any error.)
signal_binned = signal_binned.remove_background(signal_range=(138.5, 148.0))

m = signal_binned.create_model(auto_background=False)
m.components

In [ ]:
# --- Variant A: interactive ---
m.gui()

In [ ]:
# --- Variant B: in code ---
for component in m:
    print(f"{component.name}   active={component.active}")

In [ ]:
m.plot()

In [ ]:
m.multifit(kind="smart")

In [ ]:
m.plot_results()

## 4. Fine structure of the Ca-L2,3 edge

**Deviation from the original:** it used the Si references from the nanopore
dataset (90-174 eV). Those overlap with this data (from 138 eV) by only about
one third.

This dataset has its own references: the `Ca Standards` folder holds CaCO3,
CaO, Ca(OH)2, CaF2, CaSO4 and metallic Ca. With those, the fine structure of
the Ca-L2,3 edge tells you which chemical form the calcium is in.

In [ ]:
# Window around the Ca-L2,3 edge (346 eV). The background is fitted between
# the C-K edge (284 eV) and Ca (346 eV).
signal_binned = signal.rebin(scale=[2, 2, 1])
signal_binned = signal_binned.remove_background(signal_range=(300.0, 340.0))
signal_binned = signal_binned.isig[340.0:400.0]

s_smooth = signal_binned.deepcopy()
s_smooth.data = gaussian_filter1d(s_smooth.data, sigma=2, axis=-1)
s_smooth.plot()

In [ ]:
standards = load_standards("ca_standards", sigma=2)

# The references are raw spectra. They need exactly the same preprocessing as
# the data - otherwise you fit background-carrying curves against
# background-free data.
for name, s in list(standards.items()):
    s.set_signal_type("EELS")
    s = s.remove_background(signal_range=(300.0, 340.0))
    s = s.isig[340.0:400.0]
    s.data = s.data / s.data.max()
    standards[name] = s
    print(f"{name:16s} {s.axes_manager[-1].size} channels")

In [ ]:
# auto_add_edges=False: the six references already describe the Ca edge.
# Additional Ca edge components would model the same thing twice and make
# the fit ambiguous.
m = s_smooth.create_model(auto_background=False, auto_add_edges=False)

for name, s in standards.items():
    pattern = hs.model.components1D.ScalableFixedPattern(s)
    pattern.name = name
    pattern.xscale.free = False
    pattern.shift.free = False
    pattern.yscale.bmin = 0
    pattern.yscale.bmax = 1e7
    m.append(pattern)

m.components

In [ ]:
# --- Variant A: interactive ---
m.gui()

In [ ]:
# --- Variant B: same thing in code ---
# The model holds two kinds of component: the edge components (EELSCLEdge)
# created by add_elements, and our reference patterns. Only the latter have
# a yscale, so ask before reading it.
for component in m:
    yscale = getattr(component, "yscale", None)
    if yscale is None:
        print(f"{component.name:20s} (edge model, no yscale)")
    else:
        print(f"{component.name:20s} yscale={yscale.value}")

In [ ]:
m.plot()

In [ ]:
m.multifit(bounded=True)

In [ ]:
m.plot_results()

## Exercises

1. Set `rebin` to `[4, 4, 1]`. How much longer does the fit take, and do you
   actually see more in the elemental maps?
2. Remove individual elements from `add_elements`. For which does the fit get
   visibly worse, for which does almost nothing change - and what does that tell you?
3. Look at which of the six Ca references gets the largest share. Does that match
   what you know about the sample?